# Step 3: map the reads and build the consensus sequence

Step 2 gave us clean reads. This notebook turns them into one genome sequence: the **consensus**.

**Mapping** means placing every read onto a known **reference genome**, the way you would place puzzle pieces onto the picture on the box. Where the reads pile up, the most common base at each position is the consensus base. Positions with too few reads become `N` (unknown).

This is the standard method for SARS-CoV-2 and is what ENA, Galaxy and nf-core all use for this type of data. The commands are the same ones ENA's own SARS-CoV-2 Nanopore pipeline runs ([enasequence/covid-sequence-analysis-workflow](https://github.com/enasequence/covid-sequence-analysis-workflow), part of the BY-COVID WorkflowHub collection).

What it does, in order:

1. Asks: **which virus is it?** It maps one file against every virus in NCBI RefSeq, so nothing has to be known in advance.
2. Downloads one reference genome, the one the first step points to.
3. Maps every clean file to it.
4. Reports the **percentage of reads that mapped**. This is also the check for leftover human reads.
5. Reports how much of the genome is covered and how deeply.
6. Writes the consensus sequence, one FASTA file per run. **This is the final result of the use case.**
7. Writes a summary table.

Run the cells one at a time, from top to bottom, and read the output of each before moving on.

## 1. Imports

- `csv` reads and writes tab-separated tables.
- `gzip` reads compressed files.
- `subprocess` runs another program from Python.
- `os`, `sys` and `shutil` are used once, to check the tools can be found.
- `Path` builds file and folder paths.
- `requests` fetches web addresses.

In [ ]:
import csv
import gzip
import os
import shutil
import subprocess
import sys
from pathlib import Path

import requests

## 2. Your input and the folders

- `ACCESSION`: the same as in the earlier steps.
- `REFERENCE_ACCESSION`: the reference genome to map to. `MN908947.3` is the original Wuhan SARS-CoV-2 genome, the standard reference. Section 5 checks that this is the right choice.
- `MIN_DEPTH`: a consensus base needs at least this many reads covering it, or it becomes `N`. ENA's pipeline uses 30.
- `THREADS`: how many processor threads the tools may use.

A new folder `mapping/` is made for the results. A shared folder `data/reference/` holds reference genomes, because they are not specific to one study.

In [ ]:
ACCESSION = "PRJEB40277"
REFERENCE_ACCESSION = "MN908947.3"
MIN_DEPTH = 30
THREADS = 4

DATA_FOLDER = Path("data") / ACCESSION
METADATA_FOLDER = DATA_FOLDER / "metadata"
CLEAN_FOLDER = DATA_FOLDER / "clean"
MAPPING_FOLDER = DATA_FOLDER / "mapping"
REFERENCE_FOLDER = Path("data") / "reference"

for folder in [MAPPING_FOLDER, REFERENCE_FOLDER]:
    folder.mkdir(parents=True, exist_ok=True)

print("Working in:", DATA_FOLDER)

## 3. Check the tools can be found

The first line makes sure this notebook can find programs installed in the `ls-use-case` environment. `shutil.which` gives the location of a program, or `None` if it cannot be found.

**Look at the output.** Both lines should show a location that contains `ls-use-case`. If you see `None`, choose the `ls-use-case` kernel (top right) and run the cell again.

In [ ]:
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

for tool in ["minimap2", "samtools"]:
    print(tool, "->", shutil.which(tool))

## 4. List the clean files

These are the files step 2 wrote to `clean/`. The run accession is the first part of the file name.

In [ ]:
clean_files = sorted(CLEAN_FOLDER.glob("*.clean.fastq.gz"))

for clean_file in clean_files:
    print(clean_file.name)

print()
print("Number of files:", len(clean_files))

## 5. Which virus is it?

The metadata says SARS-CoV-2, but a lab record can be wrong, and in a real outbreak the virus may not be known at all. So this section checks, without assuming.

It downloads **all virus genomes in NCBI RefSeq** (about 170 MB, once) and maps the reads from the first file against all of them together. The virus that most reads map to is the answer. This is not NCBI's own tool: it is the same `minimap2` used below, with a bigger reference.

The cell first downloads the file. If it is already on your disk it is skipped.

Lines worth explaining:

- `stream=True` and `iter_content` fetch the file in pieces of 1 MB, as in step 1.
- A file name that ends in `.gz` is accepted by `minimap2` directly.

In [ ]:
VIRAL_URL = "https://ftp.ncbi.nlm.nih.gov/refseq/release/viral/viral.1.1.genomic.fna.gz"
viral_file = REFERENCE_FOLDER / "refseq_viral.genomic.fna.gz"

if viral_file.exists():
    print("Already here, skipping:", viral_file)
else:
    print("Downloading:", VIRAL_URL)
    download = requests.get(VIRAL_URL, stream=True, timeout=60)
    download.raise_for_status()
    with open(viral_file, "wb") as output_file:
        for piece in download.iter_content(chunk_size=1024 * 1024):
            output_file.write(piece)

print("Size:", round(viral_file.stat().st_size / 1000000), "MB")

## 6. Map one file against every virus

`minimap2` takes a reference and a file of reads and writes where each read matches. The option `-x map-ont` selects the settings for Oxford Nanopore reads.

The result is written in **PAF** format: one line per match, with tab-separated columns. We need two of them:

- column 1 (`columns[0]`): the read's name
- column 6 (`columns[5]`): the virus it matched

The cell counts, for each virus, the number of different reads that matched it, and prints the top five with the full name of each virus. Next to that it prints what the ENA sample record says the virus is.

Lines worth explaining:

- `set()` is a collection that holds each item once. Adding the same read name twice keeps one copy. A read can match one virus in several places, and we want to count it once.
- `ranking.sort(reverse=True)` puts the biggest count first.
- `line.startswith(">")` finds the name line of each genome in the FASTA file.

**Look at the output.** One virus should take almost all of the reads.

In [ ]:
first_clean_file = clean_files[0]
first_run = first_clean_file.name.split(".")[0]
paf_file = MAPPING_FOLDER / f"{first_run}.refseq_viral.paf"

subprocess.run(
    ["minimap2", "-x", "map-ont", "-t", str(THREADS), "-o", str(paf_file), str(viral_file), str(first_clean_file)],
    check=True,
    stderr=subprocess.DEVNULL,
)

reads_in_file = 0
with gzip.open(first_clean_file, "rt") as input_file:
    for line in input_file:
        reads_in_file = reads_in_file + 1
reads_in_file = reads_in_file // 4

reads_per_virus = {}
with open(paf_file) as input_file:
    for line in input_file:
        columns = line.split("\t")
        read_name = columns[0]
        virus = columns[5]
        if virus not in reads_per_virus:
            reads_per_virus[virus] = set()
        reads_per_virus[virus].add(read_name)

ranking = []
for virus, read_names in reads_per_virus.items():
    ranking.append((len(read_names), virus))
ranking.sort(reverse=True)

top_five = []
for count, virus in ranking[:5]:
    top_five.append(virus)

names = {}
with gzip.open(viral_file, "rt") as input_file:
    for line in input_file:
        if line.startswith(">"):
            accession = line[1:].split(" ")[0]
            if accession in top_five:
                names[accession] = line[1:].strip()

print("Run:", first_run, " reads:", reads_in_file)
print()
for count, virus in ranking[:5]:
    print(count, "reads  ", names[virus])

with open(METADATA_FOLDER / "metadata.tsv") as input_file:
    for row in csv.DictReader(input_file, delimiter="\t"):
        if row["run_accession"] == first_run:
            print()
            print("ENA sample record says:", row["sample: scientific_name"])

## 7. Download the reference genome

The reference is fetched from ENA, using the accession you set in `REFERENCE_ACCESSION`, and saved in `data/reference/`.

**Look at the output.** The name line should say SARS-CoV-2, and the length should be about 29,900 bases. If section 6 pointed to a different virus, change `REFERENCE_ACCESSION` at the top to that virus's accession and run from section 2 again.

Lines worth explaining:

- The sequence is on every line after the first, so `"".join(lines[1:])` glues those lines together. `lines[1:]` means: all lines from the second one onwards.

In [ ]:
REFERENCE_URL = "https://www.ebi.ac.uk/ena/browser/api/fasta/" + REFERENCE_ACCESSION
reference_file = REFERENCE_FOLDER / f"{REFERENCE_ACCESSION}.fasta"

if reference_file.exists():
    print("Already here, skipping:", reference_file)
else:
    response = requests.get(REFERENCE_URL, timeout=60)
    response.raise_for_status()
    reference_file.write_text(response.text)

lines = reference_file.read_text().strip().split("\n")

print("Name line:", lines[0])
print("Length:", len("".join(lines[1:])), "bases")

## 8. Map every clean file to the reference

For each run this does two jobs in one command. `minimap2 -a` maps the reads and writes the result as **SAM**, a text format with one line per read. `|` passes that straight into `samtools sort`, which sorts the reads by their position on the genome and writes a **BAM** file, the compressed form of SAM. The `-` after it means "read from the pipe".

Then `samtools index` makes a small index file next to the BAM so tools can jump to any position.

`shell=True` is needed for `|` to work. The command is written as one piece of text, with the file names filled in by `f"..."`.

The cell then asks `samtools flagstat` how many reads there are and how many mapped.

The `reads` figure can be slightly higher than the number of reads in the clean file. A read that matches two places on the genome is counted twice.

**Look at the output.** `percent mapped` is the figure to read. A value close to 100% means almost every read in the file belongs to the virus. A low value would mean the file holds something else, for example human reads that were not removed.

Lines worth explaining:

- `stderr=subprocess.DEVNULL` hides the progress messages from minimap2, which are long.
- The percentage is inside brackets in the flagstat line, like `(100.00% : N/A)`. The two `split` calls cut out the number between `(` and `%`.

In [ ]:
summary = {}

for clean_file in clean_files:
    run = clean_file.name.split(".")[0]
    bam_file = MAPPING_FOLDER / f"{run}.bam"

    command = f"minimap2 -a -x map-ont -t {THREADS} {reference_file} {clean_file} | samtools sort -o {bam_file} -"
    subprocess.run(command, shell=True, check=True, stderr=subprocess.DEVNULL)

    subprocess.run(["samtools", "index", str(bam_file)], check=True)

    result = subprocess.run(["samtools", "flagstat", str(bam_file)], capture_output=True, text=True)

    for line in result.stdout.split("\n"):
        if "in total" in line:
            total_reads = line.split(" ")[0]
        if " mapped (" in line and "primary" not in line:
            mapped_percent = line.split("(")[1].split("%")[0]

    summary[run] = {"reads": total_reads, "mapped_percent": mapped_percent}

    print(run, "reads:", total_reads, " percent mapped:", mapped_percent)

## 9. How much of the genome is covered?

`samtools coverage` reports, for the reference, how much of it has at least one read (`coverage`, in percent) and how many reads sit on each position on average (`meandepth`).

The output has a header line and one data line. Both are tab-separated. The cell picks out three columns of the data line:

- `columns[3]`: number of reads
- `columns[5]`: percent of the genome covered
- `columns[6]`: mean depth

**Look at the output.** Depth is what makes a consensus trustworthy. At depth 30 or more each base is backed by at least 30 reads.

In [ ]:
for clean_file in clean_files:
    run = clean_file.name.split(".")[0]
    bam_file = MAPPING_FOLDER / f"{run}.bam"

    result = subprocess.run(["samtools", "coverage", str(bam_file)], capture_output=True, text=True)

    data_line = result.stdout.strip().split("\n")[1]
    columns = data_line.split("\t")

    summary[run]["genome_covered_percent"] = columns[5]
    summary[run]["mean_depth"] = columns[6]

    print(run, "genome covered:", columns[5], "%  mean depth:", columns[6])

## 10. Write the consensus sequence

`samtools consensus` walks along the reference and, at each position, writes the base most reads agree on.

- `-a` writes every position, including those with no reads, which become `N`.
- `-d` is the minimum depth. Positions with fewer reads than `MIN_DEPTH` become `N`.
- `-f fasta` writes FASTA format.

samtools names the sequence after the reference. That would make every run's file look the same, so the cell rewrites the name line to say which run the sequence is from and how it was made. The result is saved as `mapping/<run>.consensus.fasta`.

The cell counts the unknown bases (`N`).

Lines worth explaining:

- `raw_file.unlink()` deletes the temporary file once we have what we need.
- `.upper().count("N")` counts the letter N, whether it was written as `N` or `n`.

**Look at the output.** A good SARS-CoV-2 consensus is about 29,800 to 29,900 bases long. A few hundred `N` is normal for amplicon data, because some amplicons get fewer reads than others.

In [ ]:
for clean_file in clean_files:
    run = clean_file.name.split(".")[0]
    bam_file = MAPPING_FOLDER / f"{run}.bam"
    raw_file = MAPPING_FOLDER / f"{run}.consensus.raw.fasta"
    consensus_file = MAPPING_FOLDER / f"{run}.consensus.fasta"

    subprocess.run(
        ["samtools", "consensus", "-a", "-d", str(MIN_DEPTH), "-f", "fasta", "-o", str(raw_file), str(bam_file)],
        check=True,
    )

    lines = raw_file.read_text().strip().split("\n")
    sequence = "".join(lines[1:])
    raw_file.unlink()

    name_line = f">{run} consensus against {REFERENCE_ACCESSION} (minimap2, samtools consensus, minimum depth {MIN_DEPTH})"
    consensus_file.write_text(name_line + "\n" + sequence + "\n")

    unknown = sequence.upper().count("N")

    summary[run]["consensus_length"] = len(sequence)
    summary[run]["consensus_unknown_bases"] = unknown
    summary[run]["consensus_file"] = consensus_file.name

    print(run, "length:", len(sequence), " unknown bases (N):", unknown)

## 11. Summary table

One row per run, written to `mapping/summary.tsv`. Every number from this notebook is in it, next to the name of the consensus file it describes.

In [ ]:
columns = ["run", "reads", "mapped_percent", "genome_covered_percent", "mean_depth",
           "consensus_length", "consensus_unknown_bases", "consensus_file"]

with open(MAPPING_FOLDER / "summary.tsv", "w", newline="") as output_file:
    writer = csv.writer(output_file, delimiter="\t")
    writer.writerow(columns)
    for run, values in summary.items():
        row = [run]
        for column in columns[1:]:
            row.append(values[column])
        writer.writerow(row)
        print(row)

print()
print("Written:", MAPPING_FOLDER / "summary.tsv")

## What you have now

- `mapping/<run>.consensus.fasta`: **the consensus genome of each run.** This is the sequence the next steps use.
- `mapping/<run>.bam` and `.bai`: the reads placed on the reference.
- `mapping/summary.tsv`: reads, percent mapped, coverage, depth and unknown bases for every run.

Step 4 tries to build the genome a second way, **without** a reference. Step 5 identifies the lineage of the consensus, compares it with other records and prepares it for deposition.